In [1]:
import numpy as np
import pandas as pd

In [2]:
-(0.6*np.log2(0.6)+0.4*np.log2(0.4))
-((20/70)*np.log2(20/70)+(50/70)*np.log2(50/70))
-((20/30)*np.log2(20/30)+(10/30)*np.log2(10/30))

0.9182958340544896

In [3]:
0.971-0.88

0.09099999999999997

In [4]:
df = pd.read_csv("HR-Employee-Attrition.csv")
print(df.shape)
df.head()

(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [5]:
sample_data = pd.read_csv('https://gist.github.com/singhsidhukuldeep/ea340fa4d6eb9ff1d24252d41c476ca4/raw/3814de94fd07650775285557678048bec9dca572/sample_data.csv')
print(sample_data.shape)
sample_data.head()

(1000, 4)


,Gender,Age_less_35,JobRole,Attrition
0,Male,True,Laboratory Technician,0
1,Male,False,Sales Executive,1
2,Male,True,Sales Representative,1
3,Female,False,Healthcare Representative,0
4,Male,True,Sales Executive,0


In [6]:
def entropy(y):
    """
    y: is pandas series (basically a single column)
    """
    p = y.value_counts(normalize=True)
    # p*log(p)
    return -(np.sum(p*np.log2(p  +  1e-6))) # log of 0 is undefined

In [7]:
# pd.crosstab(sample_data.Attrition, sample_data.Gender)

In [8]:
p = sample_data.Attrition.value_counts(normalize=True)
p

Attrition
0    0.831
1    0.169
Name: proportion, dtype: float64

In [9]:
-np.sum(p*np.log2(p)), entropy(sample_data.Attrition)# entropy of the parent

(0.6554120818442417, 0.6554091964592962)

In [10]:
# feature
# example is gender
feature = sample_data.Gender
print(sample_data.Gender.unique())
# target
y = sample_data.Attrition
y[feature=='Male'] # child1
y[feature=='Female'] # child2

['Male' 'Female']


3      0
5      0
7      0
8      0
14     0
      ..
989    0
994    0
996    0
997    0
999    0
Name: Attrition, Length: 381, dtype: int64

In [11]:
print(feature.value_counts()['Male'])
print(feature.value_counts()['Female'])
feature.value_counts()

619
381


Gender
Male      619
Female    381
Name: count, dtype: int64

In [12]:
def weighted_entropy(feature, y):
    """
    y, feature: is pandas series (basically a single column)
    feature: is an output of the condition (only 2 values)
    """
    all_uniqueVals_feature = feature.unique()
    weighted_sum = 0
    for val in all_uniqueVals_feature: # eg for Gender val will be Male and then Female
        entrpy = entropy(y[feature==val])
        weight = (feature.value_counts()[val])/len(y)
        weighted_sum+= weight*entrpy
    return weighted_sum

In [13]:
0.6554120818442417

0.6554120818442417

In [14]:
weighted_entropy(sample_data.Gender, sample_data.Attrition)

0.6554058987688518

In [15]:
weighted_entropy(sample_data.Age_less_35, sample_data.Attrition)

0.6339461315402246

In [16]:
def ig(feature, y):
    parent = entropy(y)
    child = weighted_entropy(feature, y)
    return parent-child

In [17]:
ig(sample_data.Gender, sample_data.Attrition)

3.2976904443815513e-06

In [18]:
ig(sample_data.Age_less_35, sample_data.Attrition)

0.021463064919071573

In [19]:
for feature in ['Gender',	'Age_less_35',	'JobRole']:
    print(feature, ig(sample_data[feature], sample_data.Attrition))

Gender 3.2976904443815513e-06
Age_less_35 0.021463064919071573
JobRole 0.05429906804738427


In [20]:
def gini_imp(y):
    """
    y: is pandas series (basically a single column)
    """
    p = y.value_counts(normalize=True)

    return -(np.sum(p*p))